# Decision Support System (DSS)
## Context-Aware Music Recommendation & Health-Routine Optimization

**Studente:** Stefano  
**Corso:** Decision Support Systems / Sistemi Basati su Conoscenza  

---

### Abstract del Progetto
I sistemi convenzionali di raccomandazione musicale (come Spotify, Apple Music o YouTube) soffrono del problema della **"raccomandazione cieca al contesto"**: propongono canzoni basandosi sullo storico di ascolto o sulla popolarità globale, ignorando completamente **come ci sentiamo** in un dato istante.

Questo **Decision Support System (DSS)** colma questa lacuna integrando:
1. **Caratteristiche acustico-matematiche di Spotify** (BPM, valenza emotiva, livello energetico, strumentalità).
2. **Telemetria biometrica real-time da Smartwatch** (frequenza cardiaca, livello di stress su scala 0-100, attività in corso).

Attraverso la **programmazione logica (Prolog)** e regole di matching fisiologico, il DSS seleziona la traccia ottimale o genera una sessione guidata di rilassamento (*Iso-Principle*) spiegando sempre il razionale clinico della scelta.

## 1. Caricamento ed Esplorazione dei Dataset
Il sistema legge i file sorgente in `data/` contenenti il catalogo musicale e i profili biometrici.

In [ ]:
import json
from pathlib import Path

with open('data/spotify_tracks.json', 'r', encoding='utf-8') as f:
    tracks = json.load(f)
with open('data/smartwatch_biometrics.json', 'r', encoding='utf-8') as f:
    users = json.load(f)

print(f"[+] Caricati con successo {len(tracks)} brani Spotify e {len(users)} profili biometrici smartwatch.")

print("\n--- Esempio Utenti Smartwatch ---")
for u in users[:3]:
    print(f" * {u['name']} ({u['user_id']}) -> Attività: {u['current_activity']} | Battito: {u['heart_rate_bpm']} BPM | Stress: {u['stress_level']}/100")

## 2. Generazione Automatica della Knowledge Base Prolog
Lo script `generate_kb.py` traduce i file JSON in una Knowledge Base relazionale normalizzata (`knowledge_base.pl`) contenente i predicati `brano/8` e `utente/7`.

In [ ]:
# Esecuzione dell'ETL di compilazione della Knowledge Base
import generate_kb
generate_kb.main()

# Ispezione delle prime 10 righe di fatti Prolog generati
with open('knowledge_base.pl', 'r', encoding='utf-8') as f:
    righe = [next(f) for _ in range(16)]
print("".join(righe))

## 3. Le Regole del Decision Support System (`dss_rules.pl`)
Il motore di inferenza applica tre vincoli principali per filtrare i brani:
1. **Vincolo Cognitivo (Studio):** Nessun brano con parole cantate (`instrumentalness >= 0.70`), azzerando le interferenze con la memoria di lavoro verbale.
2. **Vincolo Motorio (Corsa):** Brani con `BPM >= 120` e `Energia >= 0.75` sincronizzati col passo di corsa per ridurre la fatica percepita (*Rate of Perceived Exertion*).
3. **Vincolo di Rilassamento (Stress):** Brani a basso tempo (`BPM <= 75`) e bassa energia per stimolare il tono vagale e down-regolare il sistema nervoso simpatico.

In [ ]:
from test_suite import load_data, raccomanda

tracks_map, users_map = load_data()

scenari = [
    ('user_sarah', 'CASO 1: Stress Lavorativo Acuto'),
    ('user_alex',  'CASO 2: Allenamento Corsa Cardio'),
    ('user_marco', 'CASO 3: Sessione Studio / Deep Focus')
]

for uid, titolo in scenari:
    u = users_map[uid]
    best_track, punteggio = raccomanda(uid, tracks_map, users_map)
    print("=" * 72)
    print(f" {titolo}")
    print("=" * 72)
    print(f" Utente: {u['name']} | Attività: {u['current_activity']}")
    print(f" Telemetria: Battito: {u['heart_rate_bpm']} BPM | Stress: {u['stress_level']}/100")
    print(f"\n -> BRANO RACCOMANDATO: '{best_track['title']}' di {best_track['artist']}")
    print(f" -> Punteggio Utilità: {punteggio:.1f}/100 | Genere: {best_track['genre']}")
    print(f" -> Parametri: {best_track['bpm']} BPM, Energia: {best_track['energy']}, Strumentale: {best_track['instrumentalness']}")
    print()


## 4. Altshuler's Iso-Principle (Musicoterapia Applicata)
Quando una persona ha un battito cardiaco elevato per agitazione (es. 114 BPM), passare bruscamente a un brano lento genera dissonanza affettiva e rigetto.

L'**Iso-Principle** adotta una traiettoria controllata a 3 scalini:
- **Stadio 1 (Entry / Aggancio):** Incontra l'utente sul suo tempo attuale (105 BPM).
- **Stadio 2 (Bridge / Ponte):** Inizia la decelerazione controllata (85 BPM).
- **Stadio 3 (Target / Stabilizzazione):** Guida verso la frequenza a riposo (60 BPM).

In [ ]:
from test_suite import test_iso_principle

t1, t2, t3 = test_iso_principle(tracks_map, users_map)

print("+======================================================================+")
print("| SEQUENZA MULTI-BRANO AD ENTRAINMENT (ISO-PRINCIPLE)                  |")
print("+======================================================================+")
print(f"| 1. [AGGANCIO]  {t1['title']:<28} ({t1['bpm']:.0f} BPM, E:{t1['energy']:.2f})             |")
print(f"| 2. [PONTE]     {t2['title']:<28} ({t2['bpm']:.0f} BPM, E:{t2['energy']:.2f})             |")
print(f"| 3. [OBIETTIVO] {t3['title']:<28} ({t3['bpm']:.0f} BPM, E:{t3['energy']:.2f})             |")
print("+----------------------------------------------------------------------+")
print(f"| Traiettoria: {t1['bpm']:.0f} BPM -> {t2['bpm']:.0f} BPM -> {t3['bpm']:.0f} BPM (Rallentamento controllato)|")
print("+======================================================================+")

## 5. Verifica Formale & Test Suite
Esecuzione della suite completa a 10 test di correttezza logica per validare vincoli positivi, negativi e calcolo dell'utilità.

In [ ]:
import test_suite
test_suite.main()